In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from models.rnn_model import CaptionRNN
from models.extractors import VGG19Extractor
from utils.metrics import evaluate_retrieval

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Running inference on: {device}")


In [ ]:
vocab_size = 10000  # Must match training vocabulary size

# Load trained text encoder
text_encoder = CaptionRNN(vocab_size=vocab_size).to(device)
text_encoder.load_state_dict(torch.load('../checkpoints/gru_caption_model.pth', map_location=device))
text_encoder.eval()

# Load frozen image feature extractor
vgg_extractor = VGG19Extractor().to(device)
vgg_extractor.eval()

print("Model checkpoints successfully loaded into memory.")


In [ ]:
def retrieve_top_k_images(query_tensor, query_length, image_dataset, image_features, top_k=5):
    """
    Computes similarity scores for a text query against image embeddings and visualizes top matches.
    """
    with torch.no_grad():
        query_feature = text_encoder(query_tensor.to(device), query_length.to(device))
        
    # Calculate cosine similarity across candidate image pool
    query_norm = torch.nn.functional.normalize(query_feature, p=2, dim=1)
    img_norm = torch.nn.functional.normalize(image_features, p=2, dim=1)
    similarities = torch.mm(query_norm, img_norm.t()).squeeze(0).cpu().numpy()
    
    top_indices = np.argsort(similarities)[::-1][:top_k]
    
    # Plot top retrieved images
    fig, axes = plt.subplots(1, top_k, figsize=(15, 5))
    for i, idx in enumerate(top_indices):
        img, _ = image_dataset[idx]
        axes[i].imshow(img.permute(1, 2, 0).numpy())
        axes[i].set_title(f"Rank {i+1}\nScore: {similarities[idx]:.3f}")
        axes[i].axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
# Evaluate Recall@K on test split
# Example dummy evaluation call:
dummy_text_embeds = torch.randn(100, 4096)
dummy_img_embeds = torch.randn(100, 4096)

results = evaluate_retrieval(dummy_text_embeds, dummy_img_embeds)
print("--- Evaluation Metrics ---")
for metric, score in results.items():
    print(f"{metric}: {score:.2f}%")